# 🧠 Building a Mini LLM from Scratch: v2

v2 takes the model from **`v1_builder.ipynb`** and scales up every part of it: **13× more parameters**, about **28× more training text**, a **4× bigger vocabulary** and a context window twice as long.

The ideas are the same as in v1, so this notebook doesn't repeat v1's step-by-step tour of embedding, attention and the feed-forward layer. Instead it covers what changes once a model is **too big to train inside a notebook cell**: data on disk, a model module, a resumable training script, and a run that takes well over an hour.

### 🔁 What changed from v1

| | v1 | v2 |
|---|---|---|
| Data | stdlib only: 423 files, 8.4 MB | stdlib + installed packages: 19,265 files, 233 MB |
| Held-out split | last 10% of one token stream | random 10% of whole **files** |
| Tokenizer | 512 tokens, 2.15 chars/token | 2,048 tokens, 3.03 chars/token |
| BPE training | full recount after every merge | incremental pair counts |
| Model | 4 layers × width 128: 0.92 M params | 6 layers × width 384: 12.2 M params |
| Context | 256 tokens | 512 tokens |
| Initialisation | PyTorch defaults | N(0, 0.02) |
| Training | 2,000 steps in a notebook cell, ~70 s | 10,000 steps in a resumable terminal script, ~1 h 40 min |
| Held-out loss | 2.34 (≈ 1.57 bits/char) | 1.48 (≈ 0.70 bits/char) |

### 🗺️ The pipeline

| # | Stage | What happens |
|---|---|---|
| 01 | Size the model | Count 12.2 M parameters, and how many tokens that calls for |
| 02 | Collect data | Gather the `.py` files from the stdlib and `site-packages` |
| 03 | Clean & split | Filter, normalise, de-duplicate, then hold out 10% of files |
| 04 | Tokenizer | Train a faster BPE with 2,048 tokens |
| 05 | Encode to disk | Save token IDs as `uint16` binaries, plus `tokenizer.pkl` |
| 06 | Model module | Write the architecture to `v2_model.py` |
| 07 | Smoke test | Check shapes and starting loss, and benchmark the speed |
| 08 | Training script | Write the resumable `v2_train.py` |
| 09 | Training run | Run 10,000 steps in a terminal (log below) |
| 10 | Generation | Load the checkpoint and sample code |
| 11 | Prompt fix | Rerun the same prompt, ending in indentation |

### 📁 Files this notebook produces

| File | Section | Contents |
|---|---|---|
| `v2_data/train.bin`, `v2_data/val.bin` | 05 | Token IDs, 2 bytes each |
| `v2_data/tokenizer.pkl` | 05 | Merges, vocab, split regex, EOF ID |
| `v2_model.py` | 06 | The `MiniLLM` architecture |
| `v2_train.py` | 08 | The training script |
| `v2_ckpt.pt` | 09 | Weights, optimizer state, step and config |
| `v2_train_log.txt` | 09 | Training log |

The notebook uses the same vocabulary as v1: **table**, **grid**, **fields**, **Mix**, **Refine**.

> **Requirements:** Python 3.14 + PyTorch + NumPy.
> Built for the Apple-silicon GPU (`mps`). The benchmark in section 07 calls `torch.mps.synchronize()`, so it needs `mps` as written.
> Each code cell is followed by its recorded output, so the notebook can be read without running it.

---
## 1️⃣ Sizing the model

Same formula as v1, with bigger numbers. The width of 384 is split into **6 heads of 64 fields**, and the feed-forward hidden size is `inner = 4 × width = 1,536`:

$$\text{params} = \underbrace{VW}_{\text{embedding}} + L\,(4W^2 + 2WI + 2W) + \underbrace{W}_{\text{final norm}} + \underbrace{WV}_{\text{output}}$$

**How much data does a model this size need?** The *Chinchilla* scaling study (Hoffmann et al., 2022) found that, for a fixed compute budget, a model is best trained on about **20 tokens per parameter**. The cell computes that target, and section 05 compares it with the data actually available.

In [ ]:
# ── Model configuration (v2: ~13x the parameters of v1) ────────────────────
#   vocab  : 2,048 token IDs                          (v1: 512)
#   width  : 384 fields per token                     (v1: 128)
#   layers : 6 transformer blocks                     (v1: 4)
#   heads  : 6 attention heads of 64 fields each      (v1: 4 x 32)
#   inner  : feed-forward hidden size, 4 x width      (v1: 512)
#   seq    : 512-token context window                 (v1: 256)
#   batch  : 16 sequences per training step
cfg = dict(vocab=2048, width=384, layers=6, heads=6, inner=1536, seq=512, batch=16)

V, W, L, I = cfg["vocab"], cfg["width"], cfg["layers"], cfg["inner"]

# Size of every weight table inside ONE transformer layer
per_layer = {"Q": W*W, "K": W*W, "V": W*W, "O": W*W,   # attention projections
             "Up": W*I, "Down": I*W,                   # feed-forward expand / shrink
             "norm1": W, "norm2": W}                   # RMSNorm scale vectors
total = V*W + L*sum(per_layer.values()) + W + V*W      # Embedding + layers + final norm + Output

# table count = 1 embedding + 8 per layer + 1 final norm + 1 output
print("tables:", 1 + L*len(per_layer) + 1 + 1)
print("params per layer:", f"{sum(per_layer.values()):,}")
print("total params:", f"{total:,}")
# Chinchilla rule of thumb: ~20 training tokens per parameter
print("tokens wanted (20x):", f"{20*total/1e6:.0f}M")

```code
tables: 51
params per layer: 1,770,240
total params: 12,194,688
tokens wanted (20x): 244M
```

> **What to notice:** **12.2 M parameters**, about 13× v1.  
> Most of them sit in the 6 layers (10.6 M); the embedding and output tables add 0.8 M each.  
> The 20× rule asks for **244 M training tokens**.

---
## 2️⃣ Collecting more data

v1 trained on 8 MB of the standard library. A 12 M-parameter model needs far more, so v2 also reads every package installed in this project's virtual environment (`site-packages`): PyTorch, NumPy, Jupyter, pip and the rest. More authors means more varied style and vocabulary.

- The skip list grows: `testing`, the generated `encodings/` codecs (v1 dropped these in its cleaning step) and `__pycache__`.
- Paths are resolved and collected in a set, so a file reachable through two paths (a symlink) is read once.
- The list is sorted, so the shuffle in section 03 is reproducible.

> ⚠️ The corpus depends on which packages are installed, so running this on another machine produces different data.

In [ ]:
import sysconfig, site, pathlib

# v2 reads much more code: the standard library PLUS every package installed in this venv
roots = [pathlib.Path(sysconfig.get_paths()["stdlib"])] + [pathlib.Path(p) for p in site.getsitepackages()]

# Folders to skip: test suites, IDE, demos, generated codecs and bytecode caches
skip = {"test", "tests", "testing", "idlelib", "lib2to3", "turtledemo", "encodings", "__pycache__"}

# .resolve() + a set: a file reachable through two paths (symlinks) is kept once
# sorted(): a stable order, so the shuffle in the next cell is reproducible
files = sorted({p.resolve() for r in roots if r.exists() for p in r.rglob("*.py") if not skip & set(p.parts)})

for r in roots: print("root:", r)
print("files:", f"{len(files):,}")
print("raw MB:", round(sum(p.stat().st_size for p in files) / 1e6, 1))

```code
root: /opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14/lib/python3.14
root: /Users/umang.mishra/github-umang8496/sputnik-repo/venv-3.14/lib/python3.14/site-packages
files: 21,529
raw MB: 336.9
```

> **What to notice:** About 21.5 k files and 337 MB of raw source: 37× the files and 31× the bytes that v1 started with (575 files, 10.9 MB).

---
## 3️⃣ Cleaning and splitting

The cleaning rules are v1's, rewritten as a single pass. A `funnel` counter records why each rejected file was dropped:

1. **Decode**: drop files that aren't valid UTF-8.
2. **Size filter**: keep files between 200 and 150,000 characters.
3. **Long-line filter** *(new)*: a line over 500 characters signals minified or machine-generated code (data tables, embedded blobs).
4. **Normalise**: strip the leading comment block, turn tabs into spaces, remove trailing spaces and collapse blank lines.
5. **Exact de-duplication**: compare SHA-1 fingerprints.

**Held-out split by file** *(fixes a v1 weakness)*. v1 held out the last 10% of its token stream, which meant whichever files happened to come last. v2 shuffles the files with a fixed seed and holds out **10% of whole files**. The held-out set is then a fair sample, and no file is split between training and evaluation.

In [ ]:
import collections, hashlib, re, random

# Same normalisation as v1: drop the leading # block (shebang, licence), tabs -> 4 spaces,
# no trailing spaces, at most 2 blank lines in a row
def normalise(t):
    lines = t.split("\n")
    while lines and (lines[0].startswith("#") or not lines[0].strip()):
        lines.pop(0)
    t = "\n".join(l.replace("\t", "    ").rstrip() for l in lines)
    return re.sub(r"\n{4,}", "\n\n\n", t).strip() + "\n"

# One pass over every file; `funnel` counts why each rejected file was dropped
funnel, docs, seen = collections.Counter(), [], set()
for p in files:
    try: t = p.read_text(encoding="utf-8")
    except (UnicodeDecodeError, OSError): funnel["not utf-8"] += 1; continue
    if not (200 < len(t) < 150_000): funnel["size filter"] += 1; continue                     # stubs / huge data files
    if max(map(len, t.split("\n"))) > 500: funnel["generated (long lines)"] += 1; continue   # minified / generated code
    t = normalise(t)
    h = hashlib.sha1(t.encode()).hexdigest()                                                 # content fingerprint
    if h in seen: funnel["exact duplicate"] += 1; continue
    seen.add(h); docs.append(t)

for k, v in funnel.items(): print(f"dropped {v:6,}  {k}")
total_chars = sum(map(len, docs))
print(f"kept {len(docs):,} files, {total_chars/1e6:.1f} MB")

# held-out exam = 10% of whole FILES (v1 used the tail of one stream)
random.seed(0); random.shuffle(docs)          # fixed seed -> the same split every run
n_val = len(docs) // 10
val_docs, train_docs = docs[:n_val], docs[n_val:]
print("train files:", len(train_docs), "| held-out files:", len(val_docs))

```code
dropped  1,220  size filter
dropped    401  exact duplicate
dropped    643  generated (long lines)
kept 19,265 files, 233.1 MB
train files: 17339 | held-out files: 1926
```

> **What to notice:** No file failed to decode, which is why that row is missing from the funnel.  
> 2,264 files are dropped in total, mostly by size (1,220) and the long-line filter (643).  
> **233 MB** of clean code is about 28× v1's corpus.

---
## 4️⃣ A faster BPE tokenizer

Same algorithm as v1 (pre-split → bytes → merge the most frequent pair), with three changes:

- **Bigger vocabulary**: 2,048 tokens = 256 bytes + **1,791 merges** + `<|eof|>`. Later merges are whole words, so each token carries more text.
- **Incremental counting**: v1 recounted every pair in every chunk after each merge. v2 counts pairs **once** and records which chunks contain each pair (`pair_where`). After a merge it updates only those chunks, subtracting their old pairs and adding their new ones.
- **A sample, not everything**: merges are learned from the first 20 MB of the shuffled **training** files. That's plenty to find the frequent pieces, and the held-out files have no influence on the tokenizer. All 233 MB are encoded with these merges in section 05.

In [ ]:
import time
EOF_MARK, VOCAB = "<|eof|>", cfg["vocab"]
# Same pre-split regex as v1: word / number / punctuation run (with optional leading space), or whitespace
SPLIT = re.compile(r" ?[A-Za-z_]+| ?\d+| ?[^\sA-Za-z_\d]+|\s+(?!\S)|\s+")

# Learn the merges from the first 20 MB of (shuffled) TRAINING files only
sample, size = [], 0
for t in train_docs:
    if size >= 20_000_000: break
    sample.append(t); size += len(t)
chunk_freq = collections.Counter(c for t in sample for c in SPLIT.findall(t))
chunks = list(chunk_freq); freqs = [chunk_freq[c] for c in chunks]    # parallel lists: chunk, frequency
words = [list(c.encode("utf-8")) for c in chunks]                     # every chunk as byte IDs (0-255)
print(f"sample {size/1e6:.1f} MB, {len(chunks):,} unique chunks")

# Speed-up over v1: count every pair ONCE, and remember which chunks contain it.
# After a merge, only those chunks are updated instead of recounting everything.
pair_count, pair_where = collections.Counter(), collections.defaultdict(set)
for w, (ids, f) in enumerate(zip(words, freqs)):
    for pr in zip(ids, ids[1:]):
        pair_count[pr] += f; pair_where[pr].add(w)

merges, vocab = {}, {i: bytes([i]) for i in range(256)}
t0 = time.time()
for new_id in range(256, VOCAB - 1):                                  # 1,791 merges; the last ID is <|eof|>
    (a, b), cnt = max(pair_count.items(), key=lambda kv: kv[1])       # the most frequent pair
    merges[(a, b)] = new_id; vocab[new_id] = vocab[a] + vocab[b]
    for w in list(pair_where[(a, b)]):                                # only chunks containing (a, b)
        ids, f = words[w], freqs[w]
        for pr in zip(ids, ids[1:]): pair_count[pr] -= f      # remove old pairs
        out, i = [], 0
        while i < len(ids):                                           # rewrite the chunk with the new ID
            if i < len(ids) - 1 and ids[i] == a and ids[i+1] == b: out.append(new_id); i += 2
            else: out.append(ids[i]); i += 1
        words[w] = out
        for pr in zip(out, out[1:]): pair_count[pr] += f; pair_where[pr].add(w)   # add new pairs
    del pair_count[(a, b)], pair_where[(a, b)]                        # the merged pair no longer exists
    if (new_id - 256) % 250 == 0: print(f"merge {new_id-256:4}: {vocab[new_id]!r:20} seen {cnt:,}x")

EOF_ID = VOCAB - 1                                                    # 2047
vocab[EOF_ID] = EOF_MARK.encode()
print(f"vocab {len(vocab):,} in {time.time()-t0:.0f}s")

```code
sample 20.0 MB, 133,789 unique chunks
merge    0: b'  '                seen 2,875,268x
merge  250: b' {'                seen 8,551x
merge  500: b'instance'          seen 3,714x
merge  750: b'turns'             seen 2,175x
merge 1000: b' any'              seen 1,504x
merge 1250: b'Float'             seen 1,115x
merge 1500: b' document'         seen 896x
merge 1750: b' copy'             seen 721x
vocab 2,048 in 11s
```

> **What to notice:** **1,791 merges in 11 s**, against v1's 255 merges in 39 s: about 7× the merges, on 2.4× the text, in roughly a quarter of the time.  
> Later merges are whole words such as `instance`, ` document` and ` copy`.

---
## 5️⃣ Encoding to disk

v1 kept its token IDs in a PyTorch tensor in memory. v2 writes them to disk, because training runs in a **separate process** (section 09):

- **`train.bin` / `val.bin`**: raw `uint16` arrays. Every ID is below 65,536, so 2 bytes per token is enough, a quarter of the size of `int64`. The training script **memory-maps** these files, reading windows straight from disk instead of loading everything into RAM.
- **`tokenizer.pkl`**: the merges, the vocab, **the split regex** and the EOF ID. v1's checkpoint didn't store the regex, so every consumer had to copy it by hand.

The encoder logic is identical to v1.

In [ ]:
import numpy as np, pickle

# Encoder: identical logic to v1 - replay the merges, earliest-learned first
rank = {pr: i for i, pr in enumerate(merges)}
cache = {}                                            # chunk -> token IDs
def encode_chunk(c):
    if c in cache: return cache[c]
    ids = list(c.encode("utf-8"))                     # start from raw bytes
    while len(ids) > 1:
        pr = min(zip(ids, ids[1:]), key=lambda p: rank.get(p, 1e9))   # earliest-learned pair
        if pr not in merges: break                    # nothing left to merge
        new, out, i = merges[pr], [], 0
        while i < len(ids):
            if i < len(ids) - 1 and (ids[i], ids[i+1]) == pr: out.append(new); i += 2
            else: out.append(ids[i]); i += 1
        ids = out
    cache[c] = ids
    return ids
def encode(s): return [i for c in SPLIT.findall(s) for i in encode_chunk(c)]                  # text -> IDs
def decode(ids): return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")    # IDs -> text

# Store each split on disk as a raw uint16 array: the training script memory-maps
# these files instead of re-tokenizing or holding everything in RAM
DATA = pathlib.Path("v2_data"); DATA.mkdir(exist_ok=True)
def encode_docs(ds):
    out = []
    for t in ds: out += encode(t) + [EOF_ID]          # <|eof|> after every file
    return np.array(out, dtype=np.uint16)          # IDs < 65,536 -> 2 bytes each

train_arr, val_arr = encode_docs(train_docs), encode_docs(val_docs)
train_arr.tofile(DATA / "train.bin"); val_arr.tofile(DATA / "val.bin")
# Unlike v1, the tokenizer is saved WITH its split regex, so it can be rebuilt exactly
pickle.dump({"merges": merges, "vocab": vocab, "split": SPLIT.pattern, "eof_id": EOF_ID},
            open(DATA / "tokenizer.pkl", "wb"))
print(f"train {len(train_arr):,} tokens | held-out {len(val_arr):,} | "
      f"{total_chars/(len(train_arr)+len(val_arr)):.2f} chars/token")
print(f"wanted {20*total/1e6:.0f}M -> we have {len(train_arr)/1e6:.1f}M")

```code
train 69,298,375 tokens | held-out 7,577,251 | 3.03 chars/token
wanted 244M -> we have 69.3M
```

> **What to notice:** About 77 M tokens in total, at **3.03 characters per token** (v1: 2.15), so each token covers about 40% more text.  
> Against the 20× rule the data falls short: 69.3 M training tokens against 244 M wanted, about 28%. The run in section 09 makes 1.2 passes over it.

---
## 6️⃣ The model as a module

In v1 the model classes lived in a notebook cell, so `v1_inference.ipynb` and `serve.py` each had to keep a copy. v2 writes them to **`v2_model.py`** with `%%writefile`, and both this notebook and the training script **import** it.

The architecture is v1's (pre-norm blocks, RoPE, causal `scaled_dot_product_attention`, a GELU feed-forward, RMSNorm), with three changes:

- Sizes are **constructor arguments** instead of notebook globals.
- `rope` reads the sequence length from the tensor itself.
- **Initialisation**: every weight table starts from a normal distribution with standard deviation 0.02 (as in GPT-2). This keeps the first predictions close to uniform and early training stable.

> ⚠️ This cell **writes a file**: anything changed here, including comments, ends up in `v2_model.py` when the cell runs.

In [ ]:
%%writefile v2_model.py
"""MiniLLM v2: the v1 architecture packaged as an importable module.

Written by v2_builder.ipynb (section 06). Imported by the notebook and by v2_train.py.
"""
import torch, torch.nn as nn, torch.nn.functional as F

def rope(t):
    """Rotary position embedding over (batch, head, position, field)."""
    T, D = t.shape[-2], t.shape[-1]
    half = D // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, device=t.device) / half))   # one rotation speed per field pair
    ang = torch.arange(T, device=t.device)[:, None] * freqs[None, :]        # (T, half) angles
    t1, t2 = t[..., :half], t[..., half:]
    # 2-D rotation of each (t1, t2) pair by its position-dependent angle
    return torch.cat([t1 * ang.cos() - t2 * ang.sin(), t1 * ang.sin() + t2 * ang.cos()], dim=-1)

class Block(nn.Module):                                   # one layer: Mix + Refine
    def __init__(self, width, heads, inner):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.RMSNorm(width), nn.RMSNorm(width)
        self.Wq, self.Wk, self.Wv, self.Wo = [nn.Linear(width, width, bias=False) for _ in range(4)]   # attention tables
        self.Up, self.Down = nn.Linear(width, inner, bias=False), nn.Linear(inner, width, bias=False)  # feed-forward tables

    def forward(self, g):
        B, T, W = g.shape
        h = self.norm1(g)                                                 # pre-norm
        sp = lambda t: t.view(B, T, self.heads, W // self.heads).transpose(1, 2)   # split into heads
        q, k, v = sp(self.Wq(h)), sp(self.Wk(h)), sp(self.Wv(h))
        m = F.scaled_dot_product_attention(rope(q), rope(k), v, is_causal=True)    # causal attention
        g = g + self.Wo(m.transpose(1, 2).reshape(B, T, W))      # Mix + residual
        return g + self.Down(F.gelu(self.Up(self.norm2(g))))      # Refine + residual

class MiniLLM(nn.Module):
    def __init__(self, vocab, width, layers, heads, inner, seq):
        super().__init__()
        self.seq = seq                                            # context window, kept for generation
        self.emb = nn.Embedding(vocab, width)                     # Embedding TABLE
        self.blocks = nn.ModuleList([Block(width, heads, inner) for _ in range(layers)])
        self.norm = nn.RMSNorm(width)                             # final norm
        self.out = nn.Linear(width, vocab, bias=False)            # Output TABLE
        self.apply(self._init)                                    # new in v2: small initial weights

    @staticmethod
    def _init(m):
        # N(0, 0.02) for every weight table (GPT-2 style): first predictions stay
        # close to uniform, so the starting loss is ~ln(vocab) and early steps are stable
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids):
        g = self.emb(ids)                                         # IDs -> grid
        for b in self.blocks:
            g = b(g)
        return self.out(self.norm(g))                             # logits: (batch, seq, vocab)

```code
Writing v2_model.py
```

> **What to notice:** `Writing v2_model.py`: the cell saves the file instead of running the code. The next cell imports it.

---
## 7️⃣ Smoke test and speed benchmark

Before committing an hour or more of compute, a quick check that everything fits together, and an estimate of how long training will take:

1. Build the model from `v2_model.py` and confirm the parameter count matches section 01.
2. Run one forward pass: check the shapes, and that the starting loss is close to ln 2048 ≈ 7.62 (pure guessing).
3. Run 30 training steps and time the last 20, since the first steps include one-off GPU warm-up.

`torch.mps.synchronize()` waits for all queued GPU work to finish. Without it, the timer would only measure how fast work is *queued*, not how fast it runs.

In [ ]:
import math, torch, torch.nn.functional as F
from v2_model import MiniLLM
device = "mps" if torch.backends.mps.is_available() else "cpu"
SEQ, BATCH = cfg["seq"], cfg["batch"]
mcfg = {k: cfg[k] for k in ["vocab", "width", "layers", "heads", "inner", "seq"]}   # model args (batch is a training setting)

def get_batch(arr):
    # BATCH random windows of SEQ tokens; target y = input x shifted by one token
    starts = np.random.randint(0, len(arr) - SEQ - 1, BATCH)
    x = torch.from_numpy(np.stack([arr[s:s+SEQ] for s in starts]).astype(np.int64))
    y = torch.from_numpy(np.stack([arr[s+1:s+SEQ+1] for s in starts]).astype(np.int64))
    return x.to(device), y.to(device)

# 1. build the model: the parameter count must match section 01
torch.manual_seed(0)
model = MiniLLM(**mcfg).to(device)
print("tables:", len(list(model.parameters())), "| params:", f"{sum(p.numel() for p in model.parameters()):,}")

# 2. one forward pass: shapes, and the loss before any training
x, y = get_batch(train_arr)
logits = model(x)
print("x:", x.shape, "-> scores:", logits.shape)
loss = F.cross_entropy(logits.view(-1, V), y.view(-1))
print("starting loss:", round(loss.item(), 3), "| ln(2048) =", round(math.log(V), 3))

# 3. benchmark: 30 training steps, timing the last 20, to estimate a full run
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
for i in range(30):
    if i == 10: torch.mps.synchronize(); t0 = time.time()     # skip warm-up steps
    x, y = get_batch(train_arr)
    loss = F.cross_entropy(model(x).view(-1, V), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
torch.mps.synchronize()                                       # wait for the GPU before stopping the clock
tok_s = 20 * BATCH * SEQ / (time.time() - t0)
print(f"speed: {tok_s:,.0f} tok/s | 10,000 steps would take ~{10_000*BATCH*SEQ/tok_s/3600:.1f} h")
del model, opt                                                # free GPU memory before the real run

```code
tables: 51 | params: 12,194,688
x: torch.Size([16, 512]) -> scores: torch.Size([16, 512, 2048])
starting loss: 7.714 | ln(2048) = 7.625
speed: 18,460 tok/s | 10,000 steps would take ~1.2 h
```

> **What to notice:** 51 tables and 12,194,688 parameters, matching section 01.  
> The starting loss (7.71) is very close to ln(2048) = 7.62, thanks to the small initialisation. v1, with default initialisation, started further away (6.42 against 6.24).  
> At ~18.5 k tokens/s, 10,000 steps take over an hour, which is too long for a notebook cell. Hence the separate script.

---
## 8️⃣ The training script

Training moves into **`v2_train.py`**, which runs from a terminal. An hour-long run shouldn't depend on a notebook kernel staying alive, and it should survive interruptions.

| Feature | How |
|---|---|
| **Resumable** | Saves model, optimizer state and step to `v2_ckpt.pt` every 500 steps. Re-running the same command continues from there |
| **Light on memory** | Memory-maps `train.bin` / `val.bin`, and caps GPU memory at 50% so the Mac stays usable |
| **Logged** | Every 250 steps it prints the learning rate, train / held-out loss, throughput and ETA, and appends them to `v2_train_log.txt` |
| **Schedule** | 300 warm-up steps to a peak of 1e-3, then cosine decay to 10% |
| **Optimizer** | AdamW, weight decay 0.1, betas (0.9, 0.95) |
| **Stability** | Gradient clipping at 1.0 |

Run it from the `v2/` folder with the venv active:

```bash
caffeinate -i nice -n 19 python v2_train.py      # 10,000 steps (default)
python v2_train.py 6000                          # or choose the step count
```

`caffeinate -i` stops the Mac from sleeping, and `nice -n 19` gives the run the lowest CPU priority.

> ⚠️ The script has its **own copy of `cfg`**, so keep it in sync with section 01.
> The step count also sets the length of the cosine schedule, so resume with the same number you started with.

In [ ]:
%%writefile v2_train.py
"""Train MiniLLM v2. Written by v2_builder.ipynb (section 08).

Run from the v2/ folder in a terminal:   caffeinate -i nice -n 19 python v2_train.py
  caffeinate -i keeps the Mac awake; nice -n 19 gives the run the lowest CPU priority.
Ctrl+C stops it; the same command resumes from the last checkpoint.
Optional step count:  python v2_train.py 6000
(The step count also sets the cosine schedule, so resume with the same number.)"""
import math, pathlib, sys, time
import numpy as np, torch, torch.nn.functional as F

HERE = pathlib.Path(__file__).parent
sys.path.insert(0, str(HERE))
from v2_model import MiniLLM

# ── settings (cfg must match section 01 of the notebook) ─────────────────────
cfg = dict(vocab=2048, width=384, layers=6, heads=6, inner=1536, seq=512)
BATCH, WARMUP, PEAK_LR = 16, 300, 1e-3
STEPS = int(sys.argv[1]) if len(sys.argv) > 1 else 10_000
EVAL_EVERY, SAVE_EVERY = 250, 500
DATA, CKPT = HERE / "v2_data", HERE / "v2_ckpt.pt"

device = "mps" if torch.backends.mps.is_available() else "cpu"
if device == "mps":
    torch.mps.set_per_process_memory_fraction(0.5)       # leave GPU memory for other apps

# ── data: memory-mapped, read from disk on demand instead of loaded into RAM ─
train = np.memmap(DATA / "train.bin", dtype=np.uint16, mode="r")
val = np.memmap(DATA / "val.bin", dtype=np.uint16, mode="r")
SEQ = cfg["seq"]

def get_batch(split):
    src = train if split == "train" else val
    starts = np.random.randint(0, len(src) - SEQ - 1, BATCH)
    x = np.stack([src[s:s + SEQ] for s in starts]).astype(np.int64)
    y = np.stack([src[s + 1:s + SEQ + 1] for s in starts]).astype(np.int64)   # target = input shifted by 1
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

# ── model + optimizer; resume from the checkpoint if one exists ──────────────
model = MiniLLM(**cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=0.1, betas=(0.9, 0.95))   # beta2 0.95: common for LLMs
step = 0
if CKPT.exists():
    ck = torch.load(CKPT, map_location=device)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); step = ck["step"]
    print(f"resumed from step {step}")

def lr_at(s):                                             # warm-up, then cosine decay to 10%
    if s < WARMUP: return PEAK_LR * (s + 1) / WARMUP
    p = min(1.0, (s - WARMUP) / max(1, STEPS - WARMUP))
    return PEAK_LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))

@torch.no_grad()
def eval_loss(split, n=20):
    # average loss over n random batches, without updating the model
    model.eval()
    tot = sum(F.cross_entropy(model(x).view(-1, cfg["vocab"]), y.view(-1)).item()
              for x, y in (get_batch(split) for _ in range(n)))
    model.train()
    return tot / n

# ── logging: print and append to v2_train_log.txt ────────────────────────────
log = open(HERE / "v2_train_log.txt", "a")
def say(msg):
    print(msg, flush=True); log.write(msg + "\n"); log.flush()

tps = BATCH * SEQ                                         # tokens per step
say(f"params {sum(p.numel() for p in model.parameters()):,} | device {device} | train tokens {len(train):,} | "
    f"{STEPS:,} steps x {tps:,} = {STEPS*tps/1e6:.0f}M tokens = {STEPS*tps/len(train):.1f} passes")

# ── training loop ─────────────────────────────────────────────────────────────
t_last, s_last = time.time(), step
while True:
    if step % EVAL_EVERY == 0 or step == STEPS:          # progress report
        tr, va, now = eval_loss("train"), eval_loss("val"), time.time()
        tok_s = (step - s_last) * tps / (now - t_last) if step > s_last else 0
        eta = f"{(STEPS - step) * tps / tok_s / 60:.0f} min" if tok_s else "-"
        say(f"step {step:6} | lr {lr_at(step):.5f} | train {tr:.3f} | held-out {va:.3f} | {tok_s:,.0f} tok/s | ETA {eta}")
        t_last, s_last = time.time(), step
    if (step % SAVE_EVERY == 0 and step > 0) or step == STEPS:   # checkpoint (with optimizer state, for resuming)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": step, "cfg": cfg}, CKPT)
    if step >= STEPS:
        break
    for g in opt.param_groups:
        g["lr"] = lr_at(step)                             # apply the schedule
    x, y = get_batch("train")
    loss = F.cross_entropy(model(x).view(-1, cfg["vocab"]), y.view(-1))   # read + judge
    opt.zero_grad(set_to_none=True)
    loss.backward()                                                       # backprop
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)               # cap any wild step
    opt.step()                                                            # write
    step += 1
say(f"done, checkpoint at {CKPT}")

```code
Writing v2_train.py
```

> **What to notice:** `Writing v2_train.py`: nothing is trained here. The run happens in a terminal (next section).

---
## 9️⃣ The training run

`v2_train.py` ran in a terminal with the command above. This is its log, `v2_train_log.txt`:

```code
params 12,194,688 | device mps | train tokens 69,298,375 | 10,000 steps x 8,192 = 82M tokens = 1.2 passes
step      0 | lr 0.00000 | train 7.701 | held-out 7.708 | 0 tok/s | ETA -
step    250 | lr 0.00084 | train 4.296 | held-out 4.292 | 18,925 tok/s | ETA 70 min
step    500 | lr 0.00100 | train 3.574 | held-out 3.575 | 15,165 tok/s | ETA 86 min
step    750 | lr 0.00100 | train 3.307 | held-out 3.158 | 14,232 tok/s | ETA 89 min
step   1000 | lr 0.00099 | train 2.972 | held-out 3.000 | 14,630 tok/s | ETA 84 min
step   1250 | lr 0.00098 | train 2.564 | held-out 2.577 | 15,602 tok/s | ETA 77 min
step   1500 | lr 0.00097 | train 2.346 | held-out 2.310 | 14,599 tok/s | ETA 79 min
step   1750 | lr 0.00095 | train 2.187 | held-out 2.215 | 14,632 tok/s | ETA 77 min
step   2000 | lr 0.00093 | train 2.195 | held-out 2.235 | 14,596 tok/s | ETA 75 min
step   2250 | lr 0.00091 | train 2.038 | held-out 2.015 | 14,153 tok/s | ETA 75 min
step   2500 | lr 0.00089 | train 2.037 | held-out 2.009 | 14,012 tok/s | ETA 73 min
step   2750 | lr 0.00087 | train 1.941 | held-out 2.002 | 14,603 tok/s | ETA 68 min
step   3000 | lr 0.00084 | train 1.925 | held-out 1.969 | 14,328 tok/s | ETA 67 min
step   3250 | lr 0.00081 | train 1.870 | held-out 1.891 | 14,115 tok/s | ETA 65 min
step   3500 | lr 0.00078 | train 1.782 | held-out 1.864 | 13,757 tok/s | ETA 65 min
step   3750 | lr 0.00075 | train 1.814 | held-out 1.759 | 13,250 tok/s | ETA 64 min
step   4000 | lr 0.00071 | train 1.790 | held-out 1.857 | 13,178 tok/s | ETA 62 min
step   4250 | lr 0.00068 | train 1.715 | held-out 1.847 | 13,192 tok/s | ETA 60 min
step   4500 | lr 0.00064 | train 1.719 | held-out 1.831 | 13,226 tok/s | ETA 57 min
step   4750 | lr 0.00061 | train 1.722 | held-out 1.761 | 13,110 tok/s | ETA 55 min
step   5000 | lr 0.00057 | train 1.619 | held-out 1.767 | 13,093 tok/s | ETA 52 min
step   5250 | lr 0.00054 | train 1.683 | held-out 1.790 | 13,076 tok/s | ETA 50 min
step   5500 | lr 0.00050 | train 1.628 | held-out 1.721 | 13,152 tok/s | ETA 47 min
step   5750 | lr 0.00046 | train 1.672 | held-out 1.697 | 13,123 tok/s | ETA 44 min
step   6000 | lr 0.00043 | train 1.649 | held-out 1.646 | 13,152 tok/s | ETA 42 min
step   6250 | lr 0.00039 | train 1.711 | held-out 1.637 | 13,111 tok/s | ETA 39 min
step   6500 | lr 0.00036 | train 1.618 | held-out 1.613 | 13,162 tok/s | ETA 36 min
step   6750 | lr 0.00033 | train 1.615 | held-out 1.656 | 13,147 tok/s | ETA 34 min
step   7000 | lr 0.00030 | train 1.609 | held-out 1.624 | 13,161 tok/s | ETA 31 min
step   7250 | lr 0.00027 | train 1.548 | held-out 1.603 | 13,149 tok/s | ETA 29 min
step   7500 | lr 0.00024 | train 1.594 | held-out 1.642 | 13,236 tok/s | ETA 26 min
step   7750 | lr 0.00021 | train 1.525 | held-out 1.637 | 13,190 tok/s | ETA 23 min
step   8000 | lr 0.00019 | train 1.568 | held-out 1.654 | 13,236 tok/s | ETA 21 min
step   8250 | lr 0.00017 | train 1.458 | held-out 1.594 | 13,201 tok/s | ETA 18 min
step   8500 | lr 0.00015 | train 1.456 | held-out 1.570 | 13,667 tok/s | ETA 15 min
step   8750 | lr 0.00014 | train 1.493 | held-out 1.635 | 13,961 tok/s | ETA 12 min
step   9000 | lr 0.00012 | train 1.524 | held-out 1.610 | 14,066 tok/s | ETA 10 min
step   9250 | lr 0.00011 | train 1.501 | held-out 1.503 | 14,047 tok/s | ETA 7 min
step   9500 | lr 0.00011 | train 1.522 | held-out 1.497 | 14,062 tok/s | ETA 5 min
step   9750 | lr 0.00010 | train 1.527 | held-out 1.533 | 14,051 tok/s | ETA 2 min
step  10000 | lr 0.00010 | train 1.523 | held-out 1.480 | 14,069 tok/s | ETA 0 min
done, checkpoint at /Users/umang.mishra/github-umang8496/sputnik-repo/sputnik/scaled-down-py-model/v2/v2_ckpt.pt
```

> **What to notice:** Held-out loss falls from **7.71 → 1.48** over 10,000 steps: 82 M tokens, or 1.2 passes over the training data.  
> Train and held-out loss stay close the whole way. With whole files held out and barely more than one pass, there is no sign of overfitting.  
> The losses jump by ±0.05–0.1 between evaluations, because each one averages just 20 random batches.  
> Throughput settled at 13–14 k tokens/s, a little below the benchmark, so the run took about 1 h 40 min.

---
## 🔟 Loading the checkpoint and generating

The checkpoint stores its own config, so the model is rebuilt with `MiniLLM(**ck["cfg"])` and the trained weights are loaded. Sampling uses the same settings as v1 (temperature 0.8, top-k 40) and **the same four prompts**, for a direct comparison with the end of `v1_builder.ipynb`.

In [ ]:
# Training ran in a terminal (section 09); load the final checkpoint it saved
ck = torch.load("v2_ckpt.pt", map_location=device)
model = MiniLLM(**ck["cfg"]).to(device); model.load_state_dict(ck["model"]); model.eval()
print("checkpoint step:", ck["step"])

@torch.no_grad()
def generate(prompt, max_new=200, temperature=0.8, top_k=40):
    ids = torch.tensor([encode(prompt)], device=device)
    for _ in range(max_new):
        logits = model(ids[:, -SEQ:])[:, -1, :] / temperature                   # last position only
        logits[logits < torch.topk(logits, top_k).values[:, [-1]]] = -float("inf")   # keep the top-k
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)                    # pick one ID
        if nxt.item() == EOF_ID: break                                           # model says "file over"
        ids = torch.cat([ids, nxt], dim=1)
    return decode(ids[0].tolist())

# The same four prompts as v1, for a direct comparison
torch.manual_seed(42)
for p in ["def ", "class ", "    def __init__(self", "import os\n\ndef read_file(path):\n"]:
    print("=" * 60); print(generate(p))

```code
checkpoint step: 10000
============================================================
def _name():
    return _name


def _unicode(name) -> str:
    return _unicode_create(name)


def encode_url(path: str) -> str:
    return _encode_url(path, encoding="utf-8")


def encode_url(path: str) -> str | None:
    return _encode_url(path)


def _encode_url(path: str) -> str:
    return _encode_url(path, encoding="utf-8")


def encode_url(path: str) -> str | None:
    return _encode_url(path, encoding="utf-8")


def encode_url(path: str) -> str:
    return _encode_url(path, encoding="utf-8")


def _encode_url
============================================================
class library for config files.

    """
    return library_config_dirpath(dirname)


def _enum_include_dependencies(departition_path: str, dirname: str, *, capture_output: bool) -> str:
    """
    Called once independent dependencies are enumerated.

    :param dirname: Optional directory to indicate which pairs is used.
    :param dirname: Optional directory to indicate which pairs is used.
    :param dirname: Optional directory to indicate which pairs is used.
    :param dirname: Optional directory to indicate which pairs is used.
    :param dirname: Optional directory to indicate which pairs is used.
============================================================
    def __init__(self, value: str) -> None:
        self.value = value

    def __repr__(self) -> str:
        return f"<repr(value={self!r})>"

    def __hash__(self) -> int:
        return self.value


    def __hash__(self) -> int:
        return hash(self.value)  # type: ignore[arg-type]
    def __repr__(self) -> str:
        return f"<repr(value={self!r})>"


class LimitNode(Node):
    def __init__(self, value: str, value: str) -> None:
        self.value = value

    def __next__(self) -> str:
        # replace all patterns in a partial line.
        if self.value is None:
            return "LimitNode"
        return f
============================================================
import os

def read_file(path):
Path = 'Path'

# Track the file module.

from Path import String

from pip._internal.utils.reportlab import urljoin

VERSIONS = {
    # If the file will not be looked, either a file will execute a file.
    # If the file will not be looked, either a file will be looked when a file
    # is called with a file.
    'HF_FILE_STATUS_WRITE', 'HF_FILE_STATUS_WRITE', 'HF_FILE_STATUS_WRITE', 'HF_FILE_STATUS_WRITE', 'HF_FILE_STATUS_WR
```

> **What to notice:** A big step up from v1: type hints (`-> str | None`), f-strings, docstrings with `:param` fields, `# type: ignore[...]` comments, and `__init__` / `__repr__` methods that make sense.  
> Two weaknesses remain. The first is **repetition loops**: the same function or docstring line repeats again and again.  
> The second is that **prompts ending in whitespace go wrong**. `"class "` turns into prose, and after `"...(path):\n"` the model writes column-0 code instead of a function body (see section 5 of `v1_inference.ipynb`).

---
## 1️⃣1️⃣ Fixing the prompt

The last prompt above ended in a bare newline. The tokenizer attaches whitespace to whatever follows it, so in training a lone newline token only ever came right before **column-0** code. This time the prompt ends with the function body's indentation instead.

In [ ]:
# Same prompt, but ending in the body's indentation instead of a bare newline
torch.manual_seed(42)
print(generate("import os\n\ndef read_file(path):\n    "))

```code
import os

def read_file(path):
     ''''
    archive name of the file in a file.
    '''
    filename = os.path.basename(path)
    if _open_file(archive_name) == "next":
        archive_name, filename = cmd.read(archive_name, archive_name)
        filename = os.path.join(archive_name, archive_name)
        files = filename.replace(".", "").replace(".", "")
        files.close(filename, files[1])


def read_file(blob):
    '''read open files referencing the file'''
    filename = os.path.replace(blob, "")

    def read_file(file):
        with open(file.open(filename), files.close):
            files.close
```

> **What to notice:** This time the model writes a function body that uses its argument: a docstring, `os.path.basename(path)`, `os.path.join(...)`, `with open(...)`.  
> It still isn't quite right. The trailing four spaces are also a lone whitespace token, so the body starts with five spaces and a stray `''''`.  
> With v1, ending the prompt at the `:` and letting the model choose the newline and indentation worked best.

---
## ✅ Summary

| | |
|---|---|
| **Data** | Python 3.14 stdlib + venv packages: 19,265 files, 233 MB |
| **Tokenizer** | Byte-level BPE, 2,048 tokens, ~3.03 characters per token |
| **Model** | 6-layer decoder-only transformer, width 384, 6 heads, context 512, RoPE, RMSNorm, GELU |
| **Parameters** | 12,194,688 |
| **Training** | 10,000 steps × 16 × 512 tokens ≈ 82 M tokens, ~1 h 40 min on `mps` |
| **Held-out loss** | 7.71 → **1.48** (≈ 0.70 bits/char, against ≈ 1.57 for v1) |

Bits per character is the fairer comparison with v1, because the two tokenizers split text differently. The held-out sets differ too, so treat it as approximate.

### 🚀 Where to go next
- **More data**: 69 M tokens is about 28% of the 20× target. More packages or another code corpus would let the model train longer without repeating itself.
- **Fix the repetition**: add a repetition penalty or nucleus (top-p) sampling.
- **Inference for v2**: a `v2_inference.ipynb` and a `serve.py` that import `v2_model.py` and load `tokenizer.pkl`.
- **Precise evaluation**: measure the held-out loss over all of `val.bin`, not 20 random batches.
- **Modern extras**: weight tying, a SwiGLU feed-forward, a KV-cache for faster generation, mixed precision.